<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/implementation_of_MELDER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# [CELL 1 - Dependencies & NLTK Setup]
!pip install -q mediapipe pyctcdecode nltk torch torchvision

import os
import time
import math
import sys
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from typing import List, Tuple

# Download and verify NLTK words corpus explicitly
import nltk
nltk.download('words', quiet=False)

try:
    from nltk.corpus import words
    VOCAB_WORDS = set(words.words())
except Exception as e:
    print(f"NLTK corpus load warning: {e}. Using baseline HCI/NLP dictionary.")
    VOCAB_WORDS = {"hello", "silent", "speech", "mobile", "recognizer", "phd", "research", "paper"}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"MELDER Execution Device: {device} | Dictionary size: {len(VOCAB_WORDS)} words")

NLTK corpus load warning: 
**********************************************************************
  Resource 'words' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('words')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'corpora/words'

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************
. Using baseline HCI/NLP dictionary.
MELDER Execution Device: cuda | Dictionary size: 8 words


[nltk_data] Error loading words: Security Violation [pathsec.urlopen]:
[nltk_data]     refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.


In [21]:
# [CELL 2 - Direct Lip Landmark Extractor & Overlapping Slicer]

class LipLandmarkExtractor:
    """
    Extracts normalized Lip Region of Interest (ROI) using MediaPipe's internal
    FaceMesh solution, with a robust spatial fallback.
    """
    def __init__(self, target_size: Tuple[int, int] = (64, 64)):
        self.target_size = target_size
        self.use_mp = False

        # Direct import from internal python solutions to bypass missing top-level aliases
        try:
            from mediapipe.python.solutions import face_mesh as mp_face_mesh
            self.face_mesh = mp_face_mesh.FaceMesh(
                static_image_mode=False,
                max_num_faces=1,
                refine_landmarks=True,
                min_detection_confidence=0.5,
                min_tracking_confidence=0.5
            )
            self.use_mp = True
            print("MediaPipe FaceMesh successfully initialized.")
        except Exception as e:
            print(f"MediaPipe load notice ({e}). Using mouth-region spatial crop fallback.")

        # MediaPipe 468-point mesh indices for lips
        self.LIP_INDICES = [
            61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291,
            61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291,
            78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308
        ]

    def process_frame(self, frame: np.ndarray) -> np.ndarray:
        """Processes BGR frame and outputs normalized grayscale lip ROI (64x64)."""
        h, w, _ = frame.shape

        if self.use_mp:
            try:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = self.face_mesh.process(rgb_frame)

                if results.multi_face_landmarks:
                    landmarks = results.multi_face_landmarks[0].landmark
                    lip_pts = [(int(landmarks[idx].x * w), int(landmarks[idx].y * h)) for idx in self.LIP_INDICES]

                    xs = [pt[0] for pt in lip_pts]
                    ys = [pt[1] for pt in lip_pts]
                    min_x, max_x = max(0, min(xs) - 10), min(w, max(xs) + 10)
                    min_y, max_y = max(0, min(ys) - 10), min(h, max(ys) + 10)

                    lip_crop = frame[min_y:max_y, min_x:max_x]
                    if lip_crop.size > 0:
                        lip_gray = cv2.cvtColor(lip_crop, cv2.COLOR_BGR2GRAY)
                        return cv2.resize(lip_gray, self.target_size) / 255.0
            except Exception:
                pass

        # Robust Fallback: Crop mouth area (center-lower region of frame)
        cy, cx = int(h * 0.65), w // 2
        half_w, half_h = int(w * 0.15), int(h * 0.10)
        roi = frame[max(0, cy-half_h):min(h, cy+half_h), max(0, cx-half_w):min(w, cx+half_w)]
        roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        return cv2.resize(roi_gray, self.target_size) / 255.0


class OverlappingVideoSlicer:
    """
    Slices video stream into overlapping short windows to eliminate PIPO latency.
    """
    def __init__(self, window_size: int = 15, stride: int = 5):
        self.window_size = window_size
        self.stride = stride
        self.frame_buffer = deque(maxlen=window_size)

    def add_frame(self, lip_frame: np.ndarray) -> bool:
        """Appends frame and signals when window is ready for inference."""
        self.frame_buffer.append(lip_frame)
        return len(self.frame_buffer) == self.window_size

    def get_clip(self) -> np.ndarray:
        return np.array(self.frame_buffer)

In [22]:
# [CELL 3 - MELDER Neural Network Architecture]

class LipReadingBackbone(nn.Module):
    """
    Spatiotemporal 3D-CNN + 2D ResNet Trunk + BiGRU for frame-level logit predictions.
    """
    def __init__(self, num_classes: int = 28):
        super(LipReadingBackbone, self).__init__()

        self.frontend = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=(3, 5, 5), stride=(1, 2, 2), padding=(1, 2, 2)),
            nn.BatchNorm3d(32),
            nn.PReLU(),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )

        self.trunk = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.PReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.PReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.gru = nn.GRU(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1) # (B, 1, T, H, W)
        b, c, t, h, w = x.shape

        feat = self.frontend(x)
        feat = feat.permute(0, 2, 1, 3, 4).contiguous().view(b * t, 32, feat.shape[3], feat.shape[4])
        feat = self.trunk(feat)
        feat = feat.view(b, t, -1)

        gru_out, _ = self.gru(feat)
        logits = self.fc(gru_out)
        return logits

VOCAB = ["<blank>"] + [chr(i) for i in range(ord('a'), ord('z')+1)] + [" "]
model = LipReadingBackbone(num_classes=len(VOCAB)).to(device)
print(f"MELDER Neural Backbone loaded. Vocabulary classes: {len(VOCAB)}")

MELDER Neural Backbone loaded. Vocabulary classes: 28


In [23]:
# [CELL 4 - Stream Buffer & Language Model Auto-Completion]

class MELDERStreamDecoder:
    """
    Decodes CTC logits into characters and provides real-time word auto-completion.
    """
    def __init__(self, vocab: List[str], dictionary_set: set):
        self.vocab = vocab
        self.char_buffer = ""
        self.dictionary = dictionary_set
        self.dictionary.update(["hello", "silent", "speech", "mobile", "recognizer", "phd", "research"])

    def decode_logits(self, logits: torch.Tensor) -> str:
        preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().numpy()
        decoded_chars = []
        prev_char_idx = 0

        for p in preds:
            if p != prev_char_idx and p != 0:
                decoded_chars.append(self.vocab[p])
            prev_char_idx = p

        return "".join(decoded_chars)

    def update_buffer_and_autocomplete(self, raw_chars: str) -> Tuple[str, List[str]]:
        self.char_buffer += raw_chars
        current_word = self.char_buffer.split()[-1] if self.char_buffer else ""

        suggestions = []
        if current_word:
            suggestions = [w for w in self.dictionary if w.lower().startswith(current_word.lower())][:3]

        return self.char_buffer, suggestions

In [24]:
# [CELL 5 - Full Pipeline Execution & Latency Benchmark]

def run_melder_pipeline(synthetic_video_stream: List[np.ndarray]):
    extractor = LipLandmarkExtractor(target_size=(64, 64))
    slicer = OverlappingVideoSlicer(window_size=15, stride=5)
    decoder = MELDERStreamDecoder(vocab=VOCAB, dictionary_set=VOCAB_WORDS)

    model.eval()
    print("\n--- Running Real-Time MELDER Silent Speech Inference ---")
    start_time = time.time()

    processed_clips = 0
    with torch.no_grad():
        for frame_idx, raw_frame in enumerate(synthetic_video_stream):
            lip_roi = extractor.process_frame(raw_frame)
            is_ready = slicer.add_frame(lip_roi)

            if is_ready:
                clip = slicer.get_clip()
                clip_tensor = torch.tensor(clip, dtype=torch.float32).unsqueeze(0).to(device)

                logits = model(clip_tensor)
                raw_text = decoder.decode_logits(logits)
                full_buffer, word_suggestions = decoder.update_buffer_and_autocomplete(raw_text)

                processed_clips += 1
                print(f"[Frame {frame_idx:03d} | Clip {processed_clips:02d}] "
                      f"Recognized Stream: '{full_buffer}' | "
                      f"Auto-Complete Candidates: {word_suggestions}")

    total_time = time.time() - start_time
    print(f"\n--- Execution Benchmark Summary ---")
    print(f"Total Frames Processed: {len(synthetic_video_stream)}")
    print(f"Total Temporal Window Inferences: {processed_clips}")
    print(f"Average Latency Per Window: {(total_time / max(1, processed_clips)) * 1000:.2f} ms")


# Generate synthetic video stream cleanly with torch tensors
dummy_frames = [
    torch.randint(0, 256, (480, 640, 3), dtype=torch.uint8).numpy()
    for _ in range(60)
]

# Run Pipeline
run_melder_pipeline(dummy_frames)

MediaPipe load notice (No module named 'mediapipe.python'). Using mouth-region spatial crop fallback.

--- Running Real-Time MELDER Silent Speech Inference ---
[Frame 014 | Clip 01] Recognized Stream: 'j' | Auto-Complete Candidates: []
[Frame 015 | Clip 02] Recognized Stream: 'jj' | Auto-Complete Candidates: []
[Frame 016 | Clip 03] Recognized Stream: 'jjj' | Auto-Complete Candidates: []
[Frame 017 | Clip 04] Recognized Stream: 'jjjj' | Auto-Complete Candidates: []
[Frame 018 | Clip 05] Recognized Stream: 'jjjjj' | Auto-Complete Candidates: []
[Frame 019 | Clip 06] Recognized Stream: 'jjjjjj' | Auto-Complete Candidates: []
[Frame 020 | Clip 07] Recognized Stream: 'jjjjjjj' | Auto-Complete Candidates: []
[Frame 021 | Clip 08] Recognized Stream: 'jjjjjjjj' | Auto-Complete Candidates: []
[Frame 022 | Clip 09] Recognized Stream: 'jjjjjjjjj' | Auto-Complete Candidates: []
[Frame 023 | Clip 10] Recognized Stream: 'jjjjjjjjjj' | Auto-Complete Candidates: []
[Frame 024 | Clip 11] Recognized St